# 04 — Evaluation


In [1]:
from pathlib import Path
import subprocess
import sys

import torch
from google.colab import drive


drive.mount("/content/drive")

project_dir = Path("/content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(project_dir / "requirements.txt")],
    check=True,
)

sys.path.insert(0, str(project_dir))

from common.settings import paths as get_paths

paths = get_paths()
assert torch.cuda.is_available(), "Select a Colab GPU runtime."

print("GPU:", torch.cuda.get_device_name(0))
print("Stage 1 cache:", paths["stage1_cache"])
print("Stage 2 cache:", paths["stage2_cache"])

Mounted at /content/drive
GPU: NVIDIA L4
Stage 1 cache: /content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline/preprocess/stage1
Stage 2 cache: /content/drive/MyDrive/Research/PUMA/PUMA Nuclei Pipeline/preprocess/stage2


## Validate all best checkpoints, then run nuclei-only OOF


In [2]:
from common.fast_evaluation import evaluate_all


rows = evaluate_all(
    paths["stage1_cache"],
    paths["images"],
    paths["results"],
    paths["uni2"],
    paths["nito"],
    probability_dir=paths["stage1_cache"].parent / "nito_maps",
)

print("Saved one evaluation file: results/evaluation/history.csv")

Reusing verified NiTo maps
EVAL A/convnext_tiny/fold0
Detect A/convnext_tiny/fold0: 10/52
Detect A/convnext_tiny/fold0: 20/52
Detect A/convnext_tiny/fold0: 30/52
Detect A/convnext_tiny/fold0: 40/52
Detect A/convnext_tiny/fold0: 50/52
UNI2 batch size: 256 | device: cuda
Stage 2 inference batch size: 256
Stage 2 inference batch size: 512
Stage 2 inference batch size: 512
Stage 2 inference batch size: 512
Stage 2 inference batch size: 512
Stage 2 inference batch size: 512
Classify A/convnext_tiny/fold0: 10/52
Stage 2 inference batch size: 1024
Classify A/convnext_tiny/fold0: 20/52
Classify A/convnext_tiny/fold0: 30/52
Classify A/convnext_tiny/fold0: 40/52
Classify A/convnext_tiny/fold0: 50/52
EVAL A/convnext_base/fold0
Detect A/convnext_base/fold0: 10/52
Detect A/convnext_base/fold0: 20/52
Detect A/convnext_base/fold0: 30/52
Detect A/convnext_base/fold0: 40/52
Detect A/convnext_base/fold0: 50/52
UNI2 batch size: 256 | device: cuda
Stage 2 inference batch size: 256
Stage 2 inference batch 

## Show OOF macro-F1


In [3]:
from common.summary import summarize


summarize(paths["results"])

A | convnext_tiny | official nuclei OOF macro F1=0.285999 | ROIs=205
A | convnext_base | official nuclei OOF macro F1=0.285964 | ROIs=205
B | convnext_tiny | official nuclei OOF macro F1=0.283398 | ROIs=206
B | convnext_base | official nuclei OOF macro F1=0.285622 | ROIs=206


[{'plan': 'A',
  'architecture': 'convnext_tiny',
  'fold': '0',
  'num_rois': '52',
  'tp': '19564',
  'fp': '4450',
  'fn': '4652',
  'accuracy': '0.6824809879299518',
  'precision': '0.8146914299991671',
  'recall': '0.8078956062107697',
  'f1': '0.8112792867509849',
  'macro_f1_nuclei': '0.2928134608940034',
  'roi_f1': '0.8032644662760557',
  'tumor_accuracy': '0.7906236611787747',
  'tumor_precision': '0.8683785963968809',
  'tumor_recall': '0.8982685487796398',
  'tumor_f1': '0.8830707181187408',
  'lymphocyte_accuracy': '0.685502912010074',
  'lymphocyte_precision': '0.809028422812558',
  'lymphocyte_recall': '0.8178403755868544',
  'lymphocyte_f1': '0.8134105341800524',
  'plasma_cell_accuracy': '0.4041095890410959',
  'plasma_cell_precision': '0.7375',
  'plasma_cell_recall': '0.472',
  'plasma_cell_f1': '0.575609756097561',
  'histiocyte_accuracy': '0.4004864207539522',
  'histiocyte_precision': '0.5902031063321386',
  'histiocyte_recall': '0.5547445255474452',
  'histiocyte

In [4]:
from google.colab import runtime
runtime.unassign()